In [ ]:
import ir_datasets
import csv

dataset = ir_datasets.load("msmarco-passage-v2/train")

with open("queries.tsv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["query_id", "query_text"])  # header
    for query in dataset.queries_iter():
        writer.writerow([query.query_id, query.text])

UnicodeDecodeError: 'charmap' codec can't decode byte 0x9d in position 6067: character maps to <undefined>

In [ ]:
with open("qrels.tsv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["query_id", "doc_id", "relevance"])  # header
    for qrel in dataset.qrels_iter():
        writer.writerow([qrel.query_id, qrel.doc_id, qrel.relevance])

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/castorini/anserini-tools/master/topics-and-qrels/qrels.msmarco-v2.1-doc.dev.txt"
df = pd.read_csv(url, sep=r"\s+", header=None, names=["query_id", "iteration", "doc_id", "relevance"])
df.to_csv("qrels.tsv", sep="\t", index=False)

In [ ]:
import pandas as pd
from urllib.request import urlopen

url = "https://raw.githubusercontent.com/castorini/anserini-tools/master/topics-and-qrels/topics.msmarco-v2-doc.dev.txt"

text = urlopen(url).read().decode("utf-8")
rows = []

for line in text.splitlines():
    line = line.strip()
    if not line:
        continue
    query_id, query = line.split(maxsplit=1)
    rows.append((query_id, query))

df = pd.DataFrame(rows, columns=["query_id", "query_text"])
df.to_csv("queries.tsv", sep="\t", index=False)

In [23]:
import pandas as pd

# Read the queries and qrels files, skipping lines with unexpected number of fields
queries_df = pd.read_csv("queries.tsv", sep="\t", on_bad_lines='skip')
qrels_df = pd.read_csv("qrels.tsv", sep="\t", on_bad_lines='skip')

# Merge on query_id
merged_df = queries_df.merge(qrels_df, on="query_id")

# Select only the required columns
merged_df = merged_df[["query_id", "query_text", "doc_id", "relevance"]]

# Keep only queries with relevance >= 1
merged_df = merged_df[merged_df["relevance"] >= 1]

# Filter for queries with 3+ rows
query_counts = merged_df["query_id"].value_counts()
# queries_3plus = query_counts[(query_counts >= 50) & (query_counts < 100)].index
queries_1plus = query_counts[query_counts >= 1].index
filtered_df = merged_df[merged_df["query_id"].isin(queries_1plus)]

# Write to CSV file
filtered_df.to_csv("msmarco_queries_1plus_qrels.csv", index=False)
print(f"Saved {len(filtered_df)} rows to msmarco_queries_1plus_qrels.csv")

Saved 4702 rows to msmarco_queries_1plus_qrels.csv
